# Homework 7 

## ST-554 Big data analysis

## Author: Yefrid Cordoba

In [190]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, Ridge, ElasticNet, RidgeCV,  ElasticNetCV, LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import PolynomialFeatures

## Importing the data

The data is going to be read as a `.csv` file that is in a `Data` folder relative to the working directory for this project.

In [191]:
#Importing the data separetly
r_wine = pd.read_csv('Data/winequality-red.csv', sep = ';')
w_wine = pd.read_csv('Data/winequality-white.csv', sep = ';')

A new column is added to the data to especify what type of wine (*red or white*) as a dummy variable, where:

`red = 1`\
`white = 0`

In [192]:
r_wine['type'] = [1 for i in range(len(r_wine))]
w_wine['type'] = [0 for i in range(len(w_wine))]

After the column for the wine type have been created, it is used the function `pd.concat()` to join the two datasets in just one.

In [193]:
wine_data = pd.concat([r_wine, w_wine], ignore_index = True).drop('quality', axis = 1).dropna()

In [194]:
wine_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  type                  6497 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 609.2 KB


## Split the data 

Next, the data is going to be splited between the train and test subsets for which the same proportion of red and white data is going to be picked

In [195]:
X_train, X_test, y_train, y_test = train_test_split(
  wine_data.drop("alcohol", axis = 1),
  wine_data["alcohol"],
  test_size=0.20,
  random_state = 41,
  stratify = wine_data['type']) #Setting this parameter, we ensure that the probability for the wine type is keep during the split of the data.

In [196]:
X_train['type'].value_counts()

type
0    3918
1    1279
Name: count, dtype: int64

## Regression task

### Train models

#### Multiple linear regression models

We get the means and std to do standardization of the test set.

In [197]:
std_s = X_train.apply(np.mean, axis = 0)
mean_s = X_train.apply(np.std, axis = 0)

First I will standardize the data, just the numeric variables, not the dummy variable.

In [198]:
X_train[X_train.columns.drop('type')] = X_train[X_train.columns.drop('type')].apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)

In [199]:
# Full model
cv_full_model = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = 'neg_mean_squared_error')


New columns will be created to include interaction and polynomial term of (cuadratic).

In [200]:
X_train['dens_rsugar'] = X_train['density'] * X_train['residual sugar']
X_train['rsuga_sq'] = X_train['residual sugar'] ** 2
X_train['total_free_so2'] = X_train['free sulfur dioxide'] * X_train['total sulfur dioxide']

In [201]:
#MLR model with interaction term between density and sugar.
cv_mlr1= cross_validate(
    LinearRegression(),
    X_train[['residual sugar', 'density', 'pH', 'type', 'dens_rsugar']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

#MLR model with interaction term between density and sugar.
cv_mlr2= cross_validate(
    LinearRegression(),
    X_train[['residual sugar', 'density', 'pH', 'type', 'rsuga_sq']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

#Include interaction term between the total and free sulfur dioxide.
cv_mlr3 = cross_validate(
    LinearRegression(),
    X_train[['fixed acidity','volatile acidity', 'citric acid', 'free sulfur dioxide', 'total sulfur dioxide', 'total_free_so2']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")



After getting the cross validation, the RMSE is calculated and compared to select the model with the lowest RMSE.

In [202]:
print(np.sqrt(-sum(cv_full_model['test_score'])),
      np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])))

1.1713758501325926 1.6653658732714292 1.6903611333136064 2.465408097821743


In this case the best model is the full model since the RMSE is the lowest between the models compared.

W drop the columns that were added for the interaction and polynomial terms, since those models are not going to be used later.

In [203]:
X_train = X_train.drop(['dens_rsugar', 'rsuga_sq', 'total_free_so2'], axis = 1)

Now the model is fitted and the coefficients for each varaibles are going to be calculated.

In [204]:
MLR_full = LinearRegression().fit(X_train, y_train)
print(MLR_full.intercept_)
print(np.array(list(zip(X_train.columns, MLR_full.coef_))))

10.202319785396002
[['fixed acidity' '0.703219105942735']
 ['volatile acidity' '0.1116064625074994']
 ['citric acid' '0.08252838304107193']
 ['residual sugar' '1.1377194743406738']
 ['chlorides' '-0.04337089120882551']
 ['free sulfur dioxide' '-0.05325469362721866']
 ['total sulfur dioxide' '-0.02147237132601812']
 ['density' '-2.0363842668805465']
 ['pH' '0.4345842774980316']
 ['sulphates' '0.16446554405442562']
 ['type' '1.2056925269457655']]


#### Lasso model

We use the cross-validation for the lasso on the trainig set with the same amount of folds than for the MLR models to get an optimal tunning parameter for the model.

In [205]:
lasso_cv = LassoCV(cv=5, random_state=10) \
    .fit(X_train,
         y_train)

Now we check the calculated alpha, and the coefficent for each predictor.

In [206]:
print(lasso_cv.alpha_)
print(lasso_cv.intercept_)
print(np.array(list(zip(X_train.columns, lasso_cv.coef_))))

0.0011426282612820244
10.211432935798598
[['fixed acidity' '0.6966947585401408']
 ['volatile acidity' '0.11197399704689712']
 ['citric acid' '0.08080480212862355']
 ['residual sugar' '1.120804105460362']
 ['chlorides' '-0.04247329782434617']
 ['free sulfur dioxide' '-0.05039520330133405']
 ['total sulfur dioxide' '-0.028501591264490533']
 ['density' '-2.016485355621232']
 ['pH' '0.4304380648675396']
 ['sulphates' '0.16383234467683877']
 ['type' '1.1686627828939302']]


Based on the coefficients, all of the variables are important for the model, since none of them was set to 0, considering that this wors as a variable selection operator.

Next, we fit the model using the calculated alpha on the whole trainig set.

In [207]:
lasso_full = Lasso(lasso_cv.alpha_).fit(X_train,y_train)

#### Ridge regression model

For the ridge regresssion model, we define a list of alphas and use the scoring parameter for the function to select the appropiate alpha and the coefficients for the model.

In [208]:
ridge_cv = RidgeCV( alphas = [0.1,0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 10.0],
                      scoring = 'neg_mean_squared_error',
                      cv = 5)

The model is fitted with the whole train set to get the alphas and fit the best.

In [209]:
ridge_cv.fit(X_train, y_train)

RidgeCV(alphas=[0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 10.0], cv=5,
        scoring='neg_mean_squared_error')

Now showing the values and later fitting the model with the best alpha and the training set.

In [210]:
print(ridge_cv.alpha_)
print(ridge_cv.intercept_)
print(np.array(list(zip(X_train.columns, ridge_cv.coef_))))

3.0
10.209905111703703
[['fixed acidity' '0.6984071149001206']
 ['volatile acidity' '0.1139069790638118']
 ['citric acid' '0.08301626051093142']
 ['residual sugar' '1.1261097546829124']
 ['chlorides' '-0.044489062863290935']
 ['free sulfur dioxide' '-0.05139007711468269']
 ['total sulfur dioxide' '-0.028263052162290424']
 ['density' '-2.0216212433625613']
 ['pH' '0.43232425293922505']
 ['sulphates' '0.16513936920637495']
 ['type' '1.1748708374843713']]


In [211]:
ridge_full = Ridge(ridge_cv.alpha_).fit(X_train,y_train)

#### Elastic Net model

We create an instance for the elastic net model cross-validation to run in our data set.

In [212]:
eln_cv = ElasticNetCV(cv = 5,
                      random_state = 0,
                      l1_ratio = [0.1, 0.2, 0.3, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1],
                      n_alphas = 50)

Now we fit the trainig data set to obtain the optimal alpha and L1-ratio.

In [213]:
eln_cv.fit(X_train, y_train)

ElasticNetCV(cv=5,
             l1_ratio=[0.1, 0.2, 0.3, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1],
             n_alphas=50, random_state=0)

In [214]:
print(eln_cv.alpha_)
print(eln_cv.l1_ratio_)
print(eln_cv.intercept_)
print(np.array(list(zip(X_train.columns, eln_cv.coef_))))

0.0010686581189587902
1.0
10.210838292200254
[['fixed acidity' '0.6971264349951074']
 ['volatile acidity' '0.11194936925537692']
 ['citric acid' '0.08091467579159371']
 ['residual sugar' '1.1219132430294043']
 ['chlorides' '-0.04252781393170337']
 ['free sulfur dioxide' '-0.05058383216367175']
 ['total sulfur dioxide' '-0.0280386626297219']
 ['density' '-2.0177902739221136']
 ['pH' '0.43071072727385123']
 ['sulphates' '0.16387231498825747']
 ['type' '1.1710790164987743']]


Now the model is fitted to the whole training set using the optimal alpha and L1-ratio.

In [215]:
eln_full = ElasticNet(alpha = eln_cv.alpha_,
                     l1_ratio = eln_cv.l1_ratio_)\
            .fit(X_train, y_train)

### Test models

#### RMSE

We create a function to standardize the test set before predicting and calculate the metrics to compare.

In [216]:
#quick function to standardize based off of a supplied mean and std
def my_std_fun(x, means, stds):
    return(x-means)/stds
#loop through the columns and use the function on each
for x in X_test.drop('type', axis = 1).columns:
    X_test[x] = my_std_fun(X_test[x], mean_s[x], std_s[x])
X_test.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,type
2509,0.692997,0.486958,0.177480,0.196150,0.088850,0.215708,0.763716,0.996539,0.928529,0.899646,0
5588,0.762294,-0.161293,0.396979,0.713090,0.160419,0.906265,0.416711,0.996831,0.944059,0.411243,0
4582,0.651419,0.280696,0.553763,-0.173093,-0.107967,1.004915,1.076020,0.992377,0.928529,0.354889,0
1230,0.887029,0.044969,0.616477,-0.376176,0.554053,-0.080245,0.017656,0.996992,0.996860,1.181416,1
4028,0.942467,0.516424,0.428336,0.510006,0.088850,1.728356,1.509776,0.997876,0.925423,0.749368,0


Now we calculate the RMSE for all the models.

In [217]:
mlr_pred = MLR_full.predict(X_test)
lasso_pred = lasso_full.predict(X_test)
ridge_pred = ridge_full.predict(X_test)
eln_pred = eln_full.predict(X_test)

print(np.sqrt(mean_squared_error(y_test, mlr_pred)), \
      np.sqrt(mean_squared_error(y_test, lasso_pred)),\
      np.sqrt(mean_squared_error(y_test, ridge_pred)),\
      np.sqrt(mean_squared_error(y_test, eln_pred)))

1.9525177743006568 1.9382115689777788 1.9411887445845102 1.9391465967751782


As shown by the values above, based on the RMSE, the model that fits better to the datais the lasso model, followed by the Elastic Net, mostly becuase in essence when the L1 ratio is 1, lasso model is obtained.

#### MAE

In [218]:
print(mean_absolute_error(y_test, mlr_pred), \
      mean_absolute_error(y_test, lasso_pred),\
      mean_absolute_error(y_test, ridge_pred),\
      mean_absolute_error(y_test, eln_pred))

1.5823022541670861 1.5700515677781652 1.5727457004424752 1.5708533439223615


As the best value for the also is the smallest mean absolute error, the trend is the same as when RMSE is used as metric, the lasso model is the best at predicting our response variable, followed by the elastic net model.

## Classification task

WE need to modify our training and test sets, including type as response variable, and alcohol as a predictor.

In [219]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
  wine_data.drop("type", axis = 1),
  wine_data["type"],
  test_size=0.20,
  random_state = 42,
  stratify = wine_data['type'])

### Train models

#### Logistic regression models

We get the means and std to do standardization of the test set.

In [220]:
std_s = X_train2.apply(np.mean, axis = 0)
mean_s = X_train2.apply(np.std, axis = 0)

First I will standardize the data.

In [221]:
X_train2[X_train2.columns] = X_train2[X_train2.columns].apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)

First we use the full model

In [222]:
log_reg = LogisticRegression(penalty = None, solver = 'lbfgs', max_iter = 5000)
lg1_cv = cross_validate(log_reg,
                        X_train2,
                        y_train2,
                        cv = 5,
                        scoring = 'neg_log_loss')
lg1_cv['test_score']
                        

array([-0.03891491, -0.0319875 , -0.04303053, -0.03406313, -0.0568615 ])

Now we run the a model with interaction terms.

In [223]:
log_reg = LogisticRegression(penalty = None, solver = 'lbfgs', max_iter = 5000)
poly = PolynomialFeatures(interaction_only = True, include_bias = False)
x_poly = poly.fit_transform(X_train2[['residual sugar', 'alcohol', 'density', 'pH']])
lg2_cv = cross_validate(log_reg,
                        x_poly,
                        y_train2,
                        cv = 5,
                        scoring = 'neg_log_loss')
lg2_cv['test_score']
                       

array([-0.07612475, -0.08179831, -0.07366325, -0.08779754, -0.0565888 ])

Now a model including polynomial and interaction terms with maximum degree of 2 (cuadratic interactions)

In [224]:
poly = PolynomialFeatures(degree = 2, interaction_only = False, include_bias = False)
x_poly = poly.fit_transform(X_train2[['residual sugar', 'density', 'pH', 'fixed acidity','volatile acidity', 'citric acid']])
lg3_cv = cross_validate(log_reg,
                        x_poly,
                        y_train2,
                        cv = 5,
                        scoring = 'neg_log_loss')
lg3_cv['test_score']

array([-0.09816578, -0.10237453, -0.09616872, -0.09968434, -0.06465307])

The last model to fit to compare

In [225]:
lg4_cv = cross_validate(log_reg,
                        X_train2[['chlorides', 'free sulfur dioxide', 'total sulfur dioxide']],
                        y_train2,
                        cv = 5,
                        scoring = 'neg_log_loss')
lg4_cv['test_score']

array([-0.14001171, -0.16505715, -0.1467945 , -0.15759734, -0.14086645])

Now it is time to average the negative log loss for each of the folds and compare between all the models.

In [226]:
[round(lg1_cv['test_score'].mean(), 4),
 round(lg2_cv['test_score'].mean(), 4),
 round(lg3_cv['test_score'].mean(), 4),
 round(lg4_cv['test_score'].mean(), 4)]

[-0.041, -0.0752, -0.0922, -0.1501]

We need to pic the model that has the highest negative log loss, which in this case is the model that includes all the predictors and no interaction terms.

Now we get the fitted coefficients for that model.

In [227]:
lg_best = log_reg.fit(X_train2, y_train2)

In [228]:
X_train2.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
3123,-0.469352,-0.780732,1.174549,2.666862,-0.402695,0.595512,0.753383,1.672533,0.376343,-0.962248,-1.507518
196,0.069201,1.447371,-0.129239,-0.633169,0.524351,-0.873383,-1.076958,0.706277,1.489696,0.397608,-0.249863
1704,0.069201,-1.262484,0.008002,1.881140,-0.141963,0.200041,-0.117362,0.906192,-0.118481,-1.234219,-1.088300
3417,-0.930970,-0.479637,-0.197860,-0.926506,-0.692397,-0.195431,-0.348376,-1.692702,0.994872,-0.622284,1.510853
4422,0.300010,-0.419418,0.145242,-0.088402,-0.460635,-0.703895,-1.059188,-1.279545,-0.984423,-0.350313,1.594697


In [229]:
print(log_reg.intercept_)
print(np.array(list(zip(X_train2.columns, log_reg.coef_[0]))))

[-4.52965035]
[['fixed acidity' '-0.33906742581710186']
 ['volatile acidity' '1.034996810542738']
 ['citric acid' '-0.520711532835102']
 ['residual sugar' '-5.22768876949444']
 ['chlorides' '0.7992946769959504']
 ['free sulfur dioxide' '1.340293183807756']
 ['total sulfur dioxide' '-3.061208883590289']
 ['density' '5.462486892206761']
 ['pH' '-0.22740529557360317']
 ['sulphates' '0.47616469033660336']
 ['alcohol' '2.2580221431047782']]


#### Lasso model (`L1 penalty`)

We are going to use the same parameters used before, but now setting `penalty = 'l1'`, and useig the appropiate solver.

In [230]:
log_reg_cv = LogisticRegressionCV(
    Cs=100,
    penalty = 'l1',
    solver='saga',
    cv=5,
    scoring='neg_log_loss',
    max_iter=5000
)

log_reg_cv.fit(X_train2, y_train2)
print(log_reg_cv.C_[0])


1.592282793341094


The model is fitted witht the parameters

In [231]:
lasso_reg = LogisticRegression(penalty = 'l1', C = log_reg_cv.C_[0], solver = 'saga', max_iter = 5000)
lasso_reg.fit(X_train2, y_train2)

LogisticRegression(C=1.592282793341094, max_iter=5000, penalty='l1',
                   solver='saga')

The optimal value for C is calculated and input to the model.

In [232]:
print(lasso_reg.intercept_[0])
print(np.array(list(zip(X_train2.columns, lasso_reg.coef_[0]))))

-4.335165584164411
[['fixed acidity' '-0.08718704335298905']
 ['volatile acidity' '1.0674097034146763']
 ['citric acid' '-0.485231515261119']
 ['residual sugar' '-4.57342337121967']
 ['chlorides' '0.8003362950609308']
 ['free sulfur dioxide' '1.2002037089214073']
 ['total sulfur dioxide' '-2.964213114980322']
 ['density' '4.789826802387262']
 ['pH' '-0.027608262773502958']
 ['sulphates' '0.5202814028392593']
 ['alcohol' '1.9128262110952474']]


#### Ridge regression

For this regression, the penalty will be set to 'l2', since in this case we want just to focus on the Ridge regression.

In [233]:
l2_reg_cv = LogisticRegressionCV(
    penalty = 'l2',
    solver='lbfgs',
    cv=5,
    Cs=100,
    scoring='neg_log_loss',
    max_iter=5000
)

l2_reg_cv.fit(X_train2, y_train2)
print(l2_reg_cv.C_[0])

54.62277217684348


WE used the optimal C calculated from the corss validation 

In [234]:
rr_mod = LogisticRegression(penalty = 'l2',C = l2_reg_cv.C_[0], solver = 'lbfgs', max_iter = 5000)
rr_mod.fit(X_train2, y_train2)
print(rr_mod.intercept_[0])
print(np.array(list(zip(X_train2.columns, rr_mod.coef_[0]))))

-4.504025243502191
[['fixed acidity' '-0.3130525949271957']
 ['volatile acidity' '1.0407158724681136']
 ['citric acid' '-0.5204266345936918']
 ['residual sugar' '-5.155939245589492']
 ['chlorides' '0.7978371983804843']
 ['free sulfur dioxide' '1.3266719721252607']
 ['total sulfur dioxide' '-3.0495300716087614']
 ['density' '5.389807342526918']
 ['pH' '-0.20563579017960132']
 ['sulphates' '0.48261302115526733']
 ['alcohol' '2.223223455700357']]


#### Elastic Net

The method is the same, in this case, we introduce a list of values to test different l1 ratios.

In [235]:
eln_reg_cv = LogisticRegressionCV(
    Cs=50,
    penalty = 'elasticnet',
    l1_ratios = [0.1, 0.6, 0.9, 1],
    solver='saga',
    cv=5,
    scoring='neg_log_loss',
    max_iter=5000
)

eln_reg_cv.fit(X_train2, y_train2)
print(eln_reg_cv.C_[0])
print(eln_reg_cv.l1_ratio_[0])

1.7575106248547894
1


The optimum C is 1.75, and l1_ratio is 1 for elastic net, which at the end is a lasso model.

The model is going to be fitted with the estimated parameters.

In [236]:
eln_mod = LogisticRegression(penalty = 'elasticnet',C = eln_reg_cv.C_[0],
                             l1_ratio = eln_reg_cv.l1_ratio_[0], 
                             solver = 'saga', max_iter = 5000)
eln_mod.fit(X_train2, y_train2)
print(eln_mod.intercept_[0])
print(np.array(list(zip(X_train2.columns, eln_mod.coef_[0]))))

-4.348473983099038
[['fixed acidity' '-0.10933084963494169']
 ['volatile acidity' '1.0652077001420341']
 ['citric acid' '-0.4877825733236457']
 ['residual sugar' '-4.623167490614484']
 ['chlorides' '0.7992198335480226']
 ['free sulfur dioxide' '1.2127974522026879']
 ['total sulfur dioxide' '-2.9736966554178226']
 ['density' '4.84383763628862']
 ['pH' '-0.04487136397169931']
 ['sulphates' '0.5172350357552106']
 ['alcohol' '1.9394928626721752']]


### Test models

In [237]:
for x in X_test2.columns:
    X_test2[x] = my_std_fun(X_test2[x], mean_s[x], std_s[x])
X_test2.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
2627,0.790594,0.217694,0.640681,-0.603602,0.312731,1.093253,0.620446,0.996399,1.009062,0.833316,0.877052
5599,0.707376,0.217694,1.079781,0.189475,0.330620,0.239652,0.551245,0.996500,0.884805,1.567058,0.886578
3597,0.984767,0.099925,0.734774,1.296095,0.366398,1.093253,0.983754,1.002733,0.987317,1.266035,0.734168
6483,0.721246,0.482673,0.734774,0.650567,0.241175,1.651377,1.018355,0.997224,0.925189,0.663991,0.800847
366,1.595026,1.321775,1.612974,0.189475,0.867289,-0.351302,-0.123470,1.004543,0.903444,1.096710,0.838950


#### Log-loss

First we calculte the predicted values for the type of wine based on the different models chosen based on cross-validation.

In [238]:
lg_best_pred = lg_best.predict_proba(X_test2)
lasso_pred = lasso_reg.predict_proba(X_test2)
rr_pred = rr_mod.predict_proba(X_test2)
eln_pred = eln_mod.predict_proba(X_test2)

The log loss is evaluated for all of the predicted vs actual data.

In [239]:
log_loss(y_test2, lg_best_pred)

1.6967591580078347

In [240]:
log_loss(y_test2, lasso_pred)

1.4517672421137313

In [241]:
log_loss(y_test2, rr_pred)

1.6728086956527026

In [242]:
log_loss(y_test2, eln_pred)

1.4703601341122545

And the conclusion is that the model that includes the penalty L1 in the model is the best at predicting the type of wine, since the log loss is the lowest between the model evaluated.

#### Accuracy

The same procedure is going to be done using the accuracy as metric.

In [243]:
lg_best_preda = lg_best.predict(X_test2)
lasso_preda = lasso_reg.predict(X_test2)
rr_preda = rr_mod.predict(X_test2)
eln_preda = eln_mod.predict(X_test2)

Now the accuracy score is calculated and the best model at predicting will be the ones that yields the highest accuracy score.

In [244]:
accuracy_score(y_test2, lg_best_preda)

0.5853846153846154

In [245]:
accuracy_score(y_test2, lasso_preda)

0.5915384615384616

In [246]:
accuracy_score(y_test2, rr_preda)

0.5853846153846154

In [247]:
accuracy_score(y_test2, eln_preda)

0.5915384615384616

Based on the accuracy score, the full logistic model with no penalties predicts equaly as the one tha uses the L2 penalty, and both the elastic net model and lasso model predict in the same way(noted before since they have the same L1 ratio = 1).

# Homework 8

## ST-554 Big data analysis

## Author: Yefrid Cordoba

In this part of the work, we are going to fit different types of regression trees, classification trees, and random forest to two type of responses, a numeric response (alcohol content), and a categorical response (wine type). Then those models, are going to be compared to the multiple linear regression, and logistic regression models from last homework base on two main metrics the `RMSE`, and the `MAE`.

The same dataset for training and test will be used for this part, using the same set up for `train_test_split()` function.

In [248]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

## Regression task

First the numeric variable is going to be fitted using regression tree model, and random forest model.

In [249]:
X_train3, X_test3, y_train3, y_test3 = train_test_split(
  wine_data.drop("alcohol", axis = 1),
  wine_data["alcohol"],
  test_size=0.20,
  random_state = 41,
  stratify = wine_data['type']) #Setting this parameter, we ensure that the probability for the wine type is keep during the split of the data.

### Train models

All the predictors are going to be used for the regression tree and random forest, to make a fair comparison against the `MLR` models that were fitted previously.

#### Regression tree

To select the optimal `max_depth` and `min_samples_leaf`, we create the grid of values that the `GridSearchCV()` function is going to use to look for the two values.

In [250]:
parameters = {'max_depth': range(2,15),
              'min_samples_leaf': [3, 5, 10, 50, 100] }

Now we create an instance for our grid search cross-validation using the `DecisionTreeRegressor()` model to fit the training dataset, with five folds, and using the `neg_mean_squared_error` as our metric.

In [251]:
rt_model = GridSearchCV( DecisionTreeRegressor(),
                      parameters,
                      cv = 5,
                      scoring = 'neg_mean_squared_error' )

Now we can fit the model with the `X_train3` dataset and the `y_train3` as our response variable, which contains the alcohol content of the wine.

In [252]:
rt_model.fit(X_train3, y_train3)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': range(2, 15),
                         'min_samples_leaf': [3, 5, 10, 50, 100]},
             scoring='neg_mean_squared_error')

After fitting the model, now we can get the optimal parameters for the regression tree model.

In [253]:
print(rt_model.best_estimator_)

DecisionTreeRegressor(max_depth=13, min_samples_leaf=5)


The `max_depth` parameter was the maximum value evaluated (12), while the the `min_samples_leaf` is equal to 5.

Now we fit the model to the whole training dataset.

In [254]:
reg_tree = rt_model.best_estimator_\
            .fit(X_train3, y_train3)

#### Random forest model

To perform the `GridSearchCV()` on random forest, now we need to create a dictionary now including the `max_features` as new parameter to determine, using the exact set up for cv, and scoring, however the numer of estimators will be set for 100.

In [255]:
parameters = {'max_features' : range(1, 6),
              'max_depth': [3, 4, 6, 13, 15],
              'min_samples_leaf': [3, 5, 10, 50] }

Now we create the instance using the `RandomForestRegressor()`

In [256]:
rf_model = GridSearchCV(RandomForestRegressor(n_estimators = 100),
                      parameters,
                      cv = 5,
                      scoring = 'neg_mean_squared_error' )

Now we use the training data.

In [257]:
rf_model.fit(X_train3, y_train3)

GridSearchCV(cv=5, estimator=RandomForestRegressor(),
             param_grid={'max_depth': [3, 4, 6, 13, 15],
                         'max_features': range(1, 6),
                         'min_samples_leaf': [3, 5, 10, 50]},
             scoring='neg_mean_squared_error')

In [258]:
print(rf_model.best_estimator_)

RandomForestRegressor(max_depth=15, max_features=5, min_samples_leaf=3)


Now we fit the model using the best estimated parameters.

In [259]:
ran_for = rf_model.best_estimator_\
            .fit(X_train3, y_train3)

### Test models

#### RMSE

We use the method `.predict()` for each of the models, using the respective test set (normalized or not).

In [260]:
mlr_pred = MLR_full.predict(X_test)
lasso_pred = lasso_full.predict(X_test)
ridge_pred = ridge_full.predict(X_test)
eln_pred1 = eln_full.predict(X_test)
rt_pred = reg_tree.predict(X_test3)
rf_pred = ran_for.predict(X_test3)

print(np.sqrt(mean_squared_error(y_test, mlr_pred)), \
      np.sqrt(mean_squared_error(y_test, lasso_pred)),\
      np.sqrt(mean_squared_error(y_test, ridge_pred)),\
      np.sqrt(mean_squared_error(y_test, eln_pred1)),\
      np.sqrt(mean_squared_error(y_test3, rt_pred)),\
      np.sqrt(mean_squared_error(y_test3, rf_pred)))
      

1.9525177743006568 1.9382115689777788 1.9411887445845102 1.9391465967751782 0.5057922610162103 0.39698059048467954


Using RMSE, we pick the model with the lowest value, in this case, the model that better predicts the alcohol content is the random forest model, however this model is the one with the lowest interpretability between the others, followed by the second model, which is the regression tree model.

#### MAE

Noe we repeat the evaluation for which model is better at predicting the alcohol, in this section using the `MAE`.

In [261]:
print(mean_absolute_error(y_test, mlr_pred), \
      mean_absolute_error(y_test, lasso_pred),\
      mean_absolute_error(y_test, ridge_pred),\
      mean_absolute_error(y_test, eln_pred1),\
      mean_absolute_error(y_test3, rt_pred),\
      mean_absolute_error(y_test3, rf_pred))

1.5823022541670861 1.5700515677781652 1.5727457004424752 1.5708533439223615 0.36445724779659994 0.28621977224331097


Again we pick the model that shows the lowest MAE, which is the random forest model, followed by the regression tree model.

## Classification task

In this section, classification trees, and random classification forest models are going to be fitted to predict the wine type for the whole dataset.

In [262]:
X_train4, X_test4, y_train4, y_test4 = train_test_split(
  wine_data.drop("type", axis = 1),
  wine_data["type"],
  test_size=0.20,
  random_state = 42,
  stratify = wine_data['type'])

### Train models

#### Classification tree

First we define the parameters for the grid search.

In [263]:
parameters = {'max_depth': range(2,10),
              'min_samples_leaf': [3, 5, 10, 50, 100] }

Now we create an instance of the `GridSearchCV()` for the classificacion tree model.

In [264]:
ct_model = GridSearchCV( DecisionTreeClassifier(),
                          parameters,
                          cv = 5,
                          scoring = 'neg_log_loss' )

Now we fit the model data, using the `X_train4`, and `y_train4`.

In [265]:
ct_model.fit(X_train4, y_train4)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': range(2, 10),
                         'min_samples_leaf': [3, 5, 10, 50, 100]},
             scoring='neg_log_loss')

And get the best parameters for the model.

In [266]:
print(ct_model.best_estimator_)

DecisionTreeClassifier(max_depth=4, min_samples_leaf=100)


Then, with the optimal parameters, we fit the model to the whole training set to predict and select the best model later.

In [267]:
clas_tree = ct_model.best_estimator_\
            .fit(X_train4, y_train4)

#### Random forest with classificacion trees

Now we define the parameters to look for during the grid search.

In [268]:
parameters = {'max_features' : range(1, 5),
              'max_depth': [3, 4, 6, 13, 15],
              'min_samples_leaf': [3, 5, 10, 50] }

And create an instance of the grid search.

In [269]:
rfc_model = GridSearchCV(RandomForestClassifier(n_estimators = 100),
                          parameters,
                          cv = 5,
                          scoring = 'neg_log_loss' )

And we fit the model to the training dataset from the split.

In [270]:
rfc_model.fit(X_train4, y_train4)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [3, 4, 6, 13, 15],
                         'max_features': range(1, 5),
                         'min_samples_leaf': [3, 5, 10, 50]},
             scoring='neg_log_loss')

And we get the best parameters.

In [271]:
print(rfc_model.best_estimator_)

RandomForestClassifier(max_depth=15, max_features=3, min_samples_leaf=3)


Now we fit the model.

In [272]:
Cran_for = rfc_model.best_estimator_.fit(X_train4, y_train4)

### Test models

The models are going to be evaluated using log-loss and accuracy as the two metrics for all the classification models.

#### Log-loss

We are going to use the log loss as our metric

In [273]:
lg_best_pred = lg_best.predict_proba(X_test2)
lasso_pred = lasso_reg.predict_proba(X_test2)
rr_pred = rr_mod.predict_proba(X_test2)
eln_pred = eln_mod.predict_proba(X_test2)
ct_pred = clas_tree.predict_proba(X_test4)
rfc_pred = Cran_for.predict_proba(X_test4)

And we calculate and show the log-loss for each of the models.

In [274]:
print(log_loss(y_test2, lg_best_pred),\
     log_loss(y_test2, lasso_pred),\
     log_loss(y_test2, rr_pred),\
     log_loss(y_test2, eln_pred),\
     log_loss(y_test4, ct_pred),\
     log_loss(y_test4, rfc_pred))

1.6967591580078347 1.4517672421137313 1.6728086956527026 1.4703601341122545 0.0734972326658281 0.02602992992643444


We pick the model that has the lowest log-loss. For this metric, the best model is the random forest with classification trees.

#### Accuracy

We are going to calcualte accuracy as a second metric to compare how well the models predict the wine type.

In [275]:
lg_best_preda = lg_best.predict(X_test2)
lasso_preda = lasso_reg.predict(X_test2)
rr_preda = rr_mod.predict(X_test2)
eln_preda = eln_mod.predict(X_test2)
ct_preda = clas_tree.predict(X_test4)
rfc_preda = Cran_for.predict(X_test4)

And we calculate the accuracy score for all the models and compare between them.

In [276]:
print(accuracy_score(y_test2, lg_best_preda),\
      accuracy_score(y_test2, lasso_preda),\
      accuracy_score(y_test2, rr_preda),\
      accuracy_score(y_test2, eln_preda),\
      accuracy_score(y_test4, ct_preda),\
      accuracy_score(y_test4, rfc_preda))

0.5853846153846154 0.5915384615384616 0.5853846153846154 0.5915384615384616 0.9753846153846154 0.9976923076923077


The conclusion is the same, where the random forest with classification trees is the best model at predicting the wine type between the linear and non-linear models..